# M2 VMI - TP : Medical image analysis
---

Garance MARTIN (adapted from TP_BIOMED Isabelle Bloch)

* Name: Aya
* Last name: Chihoub
* Student ID: 22513617






**This assignment is graded and must be submitted by 14 November on Moodle.** You will upload a final notebook written as a report with the presentation of methods, results and comments on your observations.

## Objectives

The aim of this practical work is to experiment various segmentation methods on medical images:

* Brain MRI for corpus callosum segmentation
* Temporal MRI sequences for heart segmentation (in particular left ventricle)
* Retinopathy images for vessel segmentation (DRIVE dataset)

For each image type, you have to apply multiple segmentation methods (if adapted):
* manual threshold
* active contours
* vesselness filters
* A simple U-Net (with Pytorch + MONAI) and a small number of epochs

And evaluate them with the appropriate evaluation methods (quantitative and qualitative).





## Load data


From your Google Drive:
* Create a new directory for the practical work
* Upload the zip file of images and unzip it
* Connect your Drive to Google Colab with

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
Working_directory = "/content/drive/MyDrive/dataTPseg/"

In [10]:
import zipfile

with zipfile.ZipFile(Working_directory + "brainMRI.zip", "r") as zip_ref:
    zip_ref.extractall(Working_directory)

### Importation of useful libraries

In [11]:
import glob
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import AxesGrid
from skimage.measure import find_contours
from scipy.spatial.distance import dice
from skimage.transform import rescale
from scipy.io import loadmat

from scipy import ndimage as ndi
from skimage import io as skio
import skimage.morphology as morpho
from skimage.draw import line
from skimage.transform import rescale
from skimage.filters import gaussian
from skimage.segmentation import active_contour
from skimage import img_as_float

# You are encouraged to add any useful library

# Brain MRI

Here you can select medial slices of the brain of 4 different subjects. You also have manual segmentations of the corpus callosum.

In [12]:
brainMRI_path = Working_directory + 'brainMRI'

listImagesbrainMRI = glob.glob(brainMRI_path + '/*-seg.png')
print('There are', len(listImagesbrainMRI), 'brain MRI images')
print(listImagesbrainMRI)

There are 4 brain MRI images
['/content/drive/MyDrive/dataTPseg/brainMRI/MRIs1-seg.png', '/content/drive/MyDrive/dataTPseg/brainMRI/MRIs2-seg.png', '/content/drive/MyDrive/dataTPseg/brainMRI/MRIs3-seg.png', '/content/drive/MyDrive/dataTPseg/brainMRI/MRIs4-seg.png']


In [13]:
# Choose a figure and plot it with the ground truth segmentation
indexIm=3 # between 0 and 3

# Brain MRI
filename_Segmentation = listImagesbrainMRI[indexIm]
print(filename_Segmentation)
im_Seg = imread(filename_Segmentation)
filename = filename_Segmentation[:-8] + '.png'
imG = imread(filename)

print('Reading image ', filename)

if imG.shape != im_Seg.shape:
  raise NameError('image and mask should have the same shape, problem...')

# In Im Seg we have masks of the corpus callosum
maskCC=im_Seg==255
contourMask = find_contours(maskCC, 0.5)

fig = plt.figure(figsize=(17, 7))
grid = AxesGrid(fig, 111,
                nrows_ncols = (1, 3),
                axes_pad = 0.5)
grid[0].imshow(imG, cmap='gray')
grid[0].axis('off')
grid[0].set_title("Original image")
grid[1].imshow(maskCC,cmap='gray')
grid[1].axis('off')
grid[1].set_title("Segmentation Mask\n Corpus Callosum")
grid[2].imshow(imG, cmap='gray')
for contour in contourMask:
  grid[2].plot(contour[:, 1], contour[:, 0], linewidth=2, c='r')
grid[2].axis('off')
grid[2].set_title("Image with segmentation\n corpus callosum")


/content/drive/MyDrive/dataTPseg/brainMRI/MRIs4-seg.png
Reading image  /content/drive/MyDrive/dataTPseg/brainMRI/MRIs4.png


Text(0.5, 1.0, 'Image with segmentation\n corpus callosum')

<Figure size 1700x700 with 6 Axes>

# Manual Thresholding
I started with manual thresholding because it's a simple segmentation method and I wanted to see if I could separate the corpus callosum from the rest of the image using only pixel intensity.

I first tried a threshold of 100 as a starting point, then I tested several values between 50 and 150 to see how the segmentation changes and find a value that keeps most of the corpus callosum without detecting too many other regions.

In [12]:
# Manual thresholding
threshold = 100

mask_threshold = imG > threshold

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(imG, cmap='gray')
plt.title("Original image")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(mask_threshold, cmap='gray')
plt.title("Manual thresholding")
plt.axis('off')

plt.show()

<Figure size 1200x500 with 2 Axes>

In [13]:
thresholds = [50, 70, 90, 110, 130, 150]

plt.figure(figsize=(15, 8))

for i, threshold in enumerate(thresholds):
    mask = imG > threshold

    plt.subplot(2, 3, i + 1)
    plt.imshow(mask, cmap='gray')
    plt.title(f"Threshold = {threshold}")
    plt.axis('off')

plt.tight_layout()
plt.show()

<Figure size 1500x800 with 6 Axes>

**Observation**

I tried different threshold values to segment the corpus callosum and I noticed that with lower thresholds, more parts of the image are detected, including regions that don't belong to the corpus callosum, but when I increase the threshold, these unwanted regions start to disappear and some parts of the corpus callosum disappear as well.

Among the values I tested, 70 seems to give the best result visually because most of the corpus callosum is still visible, even though other structures are also detected, so I can see that thresholding alone is not enough to segment it accurately.


However, I noticed that even with 70, the segmentation still includes regions outside the corpus callosum, so I decided to calculate the Dice score to evaluate the result more precisely.

In [15]:
import numpy as np

ground_truth = maskCC
predicted_mask = imG > 70

intersection = np.logical_and(ground_truth, predicted_mask).sum()

dice = (2 * intersection) / (
    ground_truth.sum() + predicted_mask.sum()
)

print("Dice score:", dice)

Dice score: 0.09806835066864784


I obtained a Dice score of 0.0981, which is very low and shows that my segmentation is quite different from the ground truth, mainly because the threshold also detects other bright structures in the image.

Since thresholding alone doesn't give a good result, I want to try restricting the segmentation to a region of interest around the corpus callosum to see if removing the unrelated parts of the image can improve the score.

# ROI
Since the Dice score with manual thresholding was very low, I decided to try using a region of interest (ROI) around the corpus callosum because most of the unwanted regions are located outside this area.

By restricting the segmentation to a smaller region, I hope to reduce the number of incorrectly detected pixels while keeping the corpus callosum.

In [16]:
plt.figure(figsize=(8, 8))
plt.imshow(imG, cmap='gray')
plt.title("Brain MRI - ROI selection")
plt.xlabel("X (pixels)")
plt.ylabel("Y (pixels)")
plt.grid()
plt.show()

print("Image shape:", imG.shape)

<Figure size 800x800 with 1 Axes>

Image shape: (512, 512)


I selected a rectangular region between x = 160 and 330 and y = 190 and 260 because this area seems to contain the corpus callosum while excluding most of the other brain structures.

I chose these coordinates by looking at the original MRI image and I left some space around the structure so that the ROI would not be too restrictive.

In [17]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

x1, x2 = 160, 330
y1, y2 = 190, 260

fig, ax = plt.subplots(figsize=(8, 8))

ax.imshow(imG, cmap='gray')

rect = patches.Rectangle(
    (x1, y1),
    x2 - x1,
    y2 - y1,
    linewidth=2,
    edgecolor='red',
    facecolor='none'
)

ax.add_patch(rect)
ax.set_title("Selected ROI around the corpus callosum")

plt.show()

<Figure size 800x800 with 1 Axes>

In [18]:
# Thresholding inside the ROI
threshold = 70

mask_roi = np.zeros_like(imG, dtype=bool)

mask_roi[y1:y2, x1:x2] = imG[y1:y2, x1:x2] > threshold

# Display results
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(imG, cmap='gray')
plt.title("Original image")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(imG > threshold, cmap='gray')
plt.title("Global thresholding")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(mask_roi, cmap='gray')
plt.title("Thresholding with ROI")
plt.axis('off')

plt.tight_layout()
plt.show()

<Figure size 1500x500 with 3 Axes>

In [19]:
ground_truth = np.asarray(maskCC) > 0

intersection = np.logical_and(ground_truth, mask_roi).sum()

dice_roi = (2 * intersection) / (
    ground_truth.sum() + mask_roi.sum()
)

print("Dice without ROI: 0.0981")
print("Dice with ROI:", round(dice_roi, 4))

Dice without ROI: 0.0981
Dice with ROI: 0.8463


After applying the same threshold inside the ROI, I obtained a Dice score of 0.8463, which is much better than the score obtained with global thresholding.

This improvement shows that the main problem with the first approach was not only the threshold value, but also the fact that the segmentation was applied to the whole image, which introduced many false positives. By restricting the segmentation to a region centered around the corpus callosum, I was able to keep the useful part of the result and remove most of the irrelevant structures.

This result shows that thresholding can work much better when I add prior information about the location of the structure, even if the method itself remains very simple.

# Active contours
After improving the thresholding result with an ROI, I wanted to try active contours because this method also takes the shape and boundaries of the structure into account instead of relying only on pixel intensity.

Since the corpus callosum has a curved shape, I thought this method could be useful for following its boundaries more accurately, so I will compare the result with thresholding to see which one works better.

In [20]:
# Prepare the image
image = img_as_float(imG)
image_smooth = gaussian(image, sigma=2)

# Define an initial elliptical contour
t = np.linspace(0, 2 * np.pi, 200)

x = 245 + 95 * np.cos(t)
y = 225 + 45 * np.sin(t)

initial_contour = np.array([y, x]).T

# Display the initial contour
plt.figure(figsize=(8, 8))
plt.imshow(imG, cmap='gray')
plt.plot(initial_contour[:, 1],
         initial_contour[:, 0], '--r', linewidth=2)
plt.title("Initial contour")
plt.axis('off')
plt.show()

<Figure size 800x800 with 1 Axes>

In [21]:
from skimage.segmentation import active_contour

# Apply active contours
snake = active_contour(
    image_smooth,
    initial_contour,
    alpha=0.01,
    beta=1,
    gamma=0.01,
    max_num_iter=2500
)

# Display the result
plt.figure(figsize=(8, 8))

plt.imshow(imG, cmap='gray')

plt.plot(initial_contour[:, 1], initial_contour[:, 0],
         '--r', label="Initial contour")

plt.plot(snake[:, 1], snake[:, 0],
         '-b', linewidth=2, label="Final contour")

plt.title("Active Contour Segmentation")
plt.legend()
plt.axis('off')
plt.show()

<Figure size 800x800 with 1 Axes>

In [22]:
from skimage.draw import polygon2mask

mask_snake = polygon2mask(imG.shape, snake)

ground_truth = np.asarray(maskCC) > 0

intersection = np.logical_and(mask_snake, ground_truth).sum()

dice_snake = (2 * intersection) / (
    mask_snake.sum() + ground_truth.sum()
)

print("Active contour Dice:", round(dice_snake, 4))
print("Thresholding + ROI Dice:", round(dice_roi, 4))

Active contour Dice: 0.2888
Thresholding + ROI Dice: 0.8463


I obtained a Dice score of 0.2888 with active contours, which is better than global thresholding but much lower than the 0.8463 obtained with thresholding and ROI.

Looking at the result, I can see that the contour moved closer to the corpus callosum but didn't follow its shape correctly, since it also included a large region underneath it. I think this is mainly because my initial elliptical contour was not well adapted to the curved shape of the corpus callosum.

For this image, thresholding with ROI gave the best result, although it required me to manually select the region where the corpus callosum is located.

### U-Net Segmentation – Brain MRI
After testing manual thresholding and active contours, I will now apply U-Net, to see how a deep learning approach compares with the previous methods.

Unlike thresholding, U-Net can learn features directly from the images and their segmentation masks, but since the dataset only contains 4 brain MRI images, I will use 3 for training and 1 for testing, with a small number of epochs as specified in the instructions.


In [2]:
!pip -q install monai

In [3]:
import torch
import torch.nn as nn

from monai.networks.nets import UNet
from monai.losses import DiceLoss

from torch.utils.data import Dataset, DataLoader

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cu130
GPU available: True


In [14]:
from skimage.transform import resize
import numpy as np
import glob

images = []
masks = []

for seg_path in sorted(glob.glob(brainMRI_path + '/*-seg.png')):

    # Load the MRI and its segmentation
    img_path = seg_path.replace('-seg.png', '.png')

    img = imread(img_path)
    mask = imread(seg_path)

    # Resize both to 128 x 128
    img = resize(img, (128, 128), preserve_range=True)
    mask = resize(mask, (128, 128),
                  order=0, preserve_range=True)

    # Normalize the image
    img = img.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    # Convert the mask to binary
    mask = (mask > 0).astype(np.float32)

    images.append(img)
    masks.append(mask)

images = np.array(images)
masks = np.array(masks)

print("Images shape:", images.shape)
print("Masks shape:", masks.shape)

Images shape: (4, 128, 128)
Masks shape: (4, 128, 128)


In [15]:
import torch

# Convert NumPy arrays to PyTorch tensors
X = torch.tensor(images, dtype=torch.float32).unsqueeze(1)
Y = torch.tensor(masks, dtype=torch.float32).unsqueeze(1)

# Split into training and testing sets
X_train, X_test = X[:3], X[3:]
Y_train, Y_test = Y[:3], Y[3:]

print("Training images:", X_train.shape)
print("Training masks:", Y_train.shape)
print("Test image:", X_test.shape)
print("Test mask:", Y_test.shape)

Training images: torch.Size([3, 1, 128, 128])
Training masks: torch.Size([3, 1, 128, 128])
Test image: torch.Size([1, 1, 128, 128])
Test mask: torch.Size([1, 1, 128, 128])


I prepared the dataset by resizing the images and masks to 128 × 128 pixels to reduce computation, then I normalized the images and converted the masks to binary values.

I kept 3 images for training and 1 for testing so I could evaluate the model on an image it had not seen during training, and I used a small 2D U-Net with Dice loss because the corpus callosum occupies only a small part of the image.

In [16]:
# Select GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create a small 2D U-Net
model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=2
).to(device)

# Loss function and optimizer
loss_function = DiceLoss(sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Device:", device)
print("U-Net model created successfully")

Device: cuda
U-Net model created successfully


In [17]:
# Training U-Net
num_epochs = 20
train_losses = []

model.train()

for epoch in range(num_epochs):
    optimizer.zero_grad()

    # Move training data to GPU
    inputs = X_train.to(device)
    labels = Y_train.to(device)

    # Forward pass
    outputs = model(inputs)

    # Calculate loss
    loss = loss_function(outputs, labels)

    # Backpropagation
    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")

Epoch 1/20, Loss: 0.9808
Epoch 2/20, Loss: 0.9746
Epoch 3/20, Loss: 0.9700
Epoch 4/20, Loss: 0.9664
Epoch 5/20, Loss: 0.9637
Epoch 6/20, Loss: 0.9620
Epoch 7/20, Loss: 0.9609
Epoch 8/20, Loss: 0.9602
Epoch 9/20, Loss: 0.9598
Epoch 10/20, Loss: 0.9594
Epoch 11/20, Loss: 0.9590
Epoch 12/20, Loss: 0.9588
Epoch 13/20, Loss: 0.9585
Epoch 14/20, Loss: 0.9583
Epoch 15/20, Loss: 0.9581
Epoch 16/20, Loss: 0.9579
Epoch 17/20, Loss: 0.9578
Epoch 18/20, Loss: 0.9576
Epoch 19/20, Loss: 0.9575
Epoch 20/20, Loss: 0.9573


In [18]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, num_epochs + 1), train_losses, marker='o')

plt.xlabel("Epoch")
plt.ylabel("Dice Loss")
plt.title("U-Net Training Loss")
plt.grid()
plt.show()

<Figure size 800x400 with 1 Axes>

I trained the U-Net for 20 epochs and observed that the Dice loss decreased from 0.9808 to 0.9573, which shows that the model is learning, but the loss remains quite high even at the end of training.

I think this could be explained by the very small dataset, since I only used 3 images for training, and also by the fact that the corpus callosum occupies a small part of each image, which makes it harder for the model to learn its segmentation.

In [19]:
# Evaluate U-Net on the test image
model.eval()

with torch.no_grad():
    output = model(X_test.to(device))
    prediction = torch.sigmoid(output)

# Convert probabilities to a binary mask
pred_mask = (prediction > 0.5).cpu().numpy()[0, 0]
true_mask = Y_test.numpy()[0, 0]

# Calculate Dice score
intersection = np.logical_and(pred_mask, true_mask).sum()
denominator = pred_mask.sum() + true_mask.sum()

dice_unet = 2 * intersection / denominator if denominator > 0 else 0

print("U-Net Dice score:", round(dice_unet, 4))

# Display the results
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(X_test[0, 0], cmap="gray")
plt.title("Original MRI")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(true_mask, cmap="gray")
plt.title("Ground truth")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(pred_mask, cmap="gray")
plt.title("U-Net prediction")
plt.axis("off")

plt.tight_layout()
plt.show()

U-Net Dice score: 0.0881


<Figure size 1200x400 with 3 Axes>

After comparing the different methods, I found that thresholding with ROI gave the best result with a Dice score of 0.8463, followed by active contours with 0.2888, while global thresholding and U-Net gave much lower scores.

The ROI helped a lot because it restricted the segmentation to the area containing the corpus callosum, while active contours struggled with its curved shape and U-Net did not have enough training data to learn it properly.

These results are based on only one test image, so I cannot conclude that thresholding is generally better than the other methods, but for this experiment it was the most effective approach.

# MRI heart

The last section is about MRI sequences of the heart. Your goal is to segment the left ventricule. Be careful, the segmentation is not a maks but a series of points (landmarks). To obtain a binary mask, you should first interpolate the points (using for instance a spline).


In [20]:
MRIheart_path = Working_directory + 'MRIheart/'


In [22]:
import zipfile

with zipfile.ZipFile(Working_directory + "MRIheart.zip", "r") as zip_ref:
    zip_ref.extractall(Working_directory)

In [23]:
import os

print(os.listdir(Working_directory + "MRIheart"))

['dataMRIheart.mat', 'segMRIheart.mat']


In [24]:
data=loadmat(MRIheart_path + 'dataMRIheart.mat')
data=data['data']
seg=loadmat(MRIheart_path + 'segMRIheart.mat')
seg=seg['seg']

print('MRI volume of the heart composed of', data.shape[2], 'slices along the z axis and', data.shape[3],
'temporal frames. Each slice is an image ', data.shape[0], ' x ',  data.shape[1])
print('For each slice and at each time frame we have a manual segmentation composed of',seg[4,4].shape[0] , '2D landmarks')

print('Be careful, some slices do not have the left ventricle and the manual segmentation is not simply empty but it contains the value:', seg[0,0] )

MRI volume of the heart composed of 11 slices along the z axis and 20 temporal frames. Each slice is an image  256  x  256
For each slice and at each time frame we have a manual segmentation composed of 65 2D landmarks
Be careful, some slices do not have the left ventricle and the manual segmentation is not simply empty but it contains the value: [[-99999]]


In [25]:
plt.figure(figsize=(20, 9))
plt.suptitle('Different slices at the same time frame with the red manual segmentation at the bottom')
plt.subplot(2, 5, 1)
plt.imshow(data[:,:,4,1],cmap="gray")
plt.gca().invert_yaxis()

plt.subplot(2, 5, 2)
plt.imshow(data[:,:,5,1],cmap="gray")
plt.gca().invert_yaxis()

plt.subplot(2, 5, 3)
plt.imshow(data[:,:,6,1],cmap="gray")
plt.gca().invert_yaxis()

plt.subplot(2, 5, 4)
plt.imshow(data[:,:,7,1],cmap="gray")
plt.gca().invert_yaxis()

plt.subplot(2, 5, 5)
plt.imshow(data[:,:,8,1],cmap="gray")
plt.gca().invert_yaxis()

plt.subplot(2, 5, 6)
plt.imshow(data[:,:,4,1],cmap="gray")
plt.gca().invert_yaxis()
plt.scatter(seg[4,1][:,0], seg[4,1][:,1], c='r',alpha=0.1)

plt.subplot(2, 5, 7)
plt.imshow(data[:,:,5,1],cmap="gray")
plt.gca().invert_yaxis()
plt.scatter(seg[5,1][:,0], seg[5,1][:,1], c='r',alpha=0.1)

plt.subplot(2, 5, 8)
plt.imshow(data[:,:,6,1],cmap="gray")
plt.gca().invert_yaxis()
plt.scatter(seg[6,1][:,0], seg[6,1][:,1], c='r',alpha=0.1)

plt.subplot(2, 5, 9)
plt.imshow(data[:,:,7,1],cmap="gray")
plt.gca().invert_yaxis()
plt.scatter(seg[7,1][:,0], seg[7,1][:,1], c='r',alpha=0.1)

plt.subplot(2, 5, 10)
plt.imshow(data[:,:,8,1],cmap="gray")
plt.gca().invert_yaxis()
plt.scatter(seg[8,1][:,0], seg[8,1][:,1], c='r',alpha=0.1)

<Figure size 2000x900 with 10 Axes>

In [27]:
print("MRI shape:", data.shape)
print("Landmarks array shape:", seg.shape)

# Examine the landmarks of one slice and time frame
print("Landmarks shape:", seg[4, 4].shape)
print("First 5 landmarks:")
print(seg[4, 4][:5])

MRI shape: (256, 256, 11, 20)
Landmarks array shape: (11, 20)
Landmarks shape: (65, 2)
First 5 landmarks:
[[121.08597112 125.31241268]
 [118.27062829 126.11911748]
 [115.72074619 127.53333429]
 [113.61216305 129.57394424]
 [112.19063444 132.12545465]]


In [28]:
# Select a slice and time frame
slice_idx = 4
time_idx = 4

image_heart = data[:, :, slice_idx, time_idx]
landmarks = seg[slice_idx, time_idx]

# Separate the two contours
contour1 = landmarks[:32]
contour2 = landmarks[33:]

plt.figure(figsize=(7, 7))
plt.imshow(image_heart, cmap="gray")
plt.plot(contour1[:, 0], contour1[:, 1], 'r.-', label="Contour 1")
plt.plot(contour2[:, 0], contour2[:, 1], 'b.-', label="Contour 2")
plt.gca().invert_yaxis()
plt.title("Left ventricle landmarks")
plt.legend()
plt.show()

<Figure size 700x700 with 1 Axes>

The manual segmentation is given as two sets of landmarks, one for the inner contour and one for the outer contour of the left ventricle, so I need to convert these points into a binary mask before using them to evaluate the segmentation methods.

I will fill the area inside the outer contour and remove the area inside the inner contour, since I want to keep only the region corresponding to the myocardium.

In [30]:
from skimage.draw import polygon2mask

# Create binary masks from the two contours
mask_inner = polygon2mask(
    image_heart.shape,
    contour1[:, ::-1]
)

mask_outer = polygon2mask(
    image_heart.shape,
    contour2[:, ::-1]
)

# Keep the region between the contours
mask_heart = mask_outer & ~mask_inner

# Display the result
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(image_heart, cmap="gray")
plt.title("Original MRI")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mask_heart, cmap="gray")
plt.title("Binary mask - Myocardium")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(image_heart, cmap="gray")
plt.contour(mask_heart, colors="red", linewidths=1)
plt.title("MRI with segmentation")
plt.axis("off")

plt.tight_layout()
plt.show()

<Figure size 1200x400 with 3 Axes>

After converting the landmarks into a binary mask, I obtained a ring-shaped region corresponding to the myocardium, with the area inside the inner contour removed.

When I overlay the mask on the original MRI, both boundaries seem to follow the structure correctly, so I can now use this mask as the ground truth to evaluate the segmentation methods.

In [31]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import torch

from PIL import Image
from skimage.draw import polygon2mask
from skimage.filters import gaussian, frangi, threshold_otsu
from skimage.segmentation import active_contour
from skimage.morphology import remove_small_objects
from skimage.transform import resize
from torch.utils.data import DataLoader, TensorDataset
from monai.networks.nets import UNet
from monai.losses import DiceLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

np.random.seed(42)
torch.manual_seed(42)

print("Device:", device)

# Dice score
def dice_score(pred, truth, valid=None):
    pred = np.asarray(pred, dtype=bool)
    truth = np.asarray(truth, dtype=bool)

    if valid is not None:
        valid = np.asarray(valid, dtype=bool)
        pred = pred[valid]
        truth = truth[valid]

    denominator = pred.sum() + truth.sum()

    if denominator == 0:
        return 1.0

    intersection = np.logical_and(pred, truth).sum()
    return float(2 * intersection / denominator)


# Normalize an image between 0 and 1
def scale01(img):
    img = np.asarray(img, dtype=np.float32)
    return (img - img.min()) / (np.ptp(img) + 1e-8)


# Convert the 65 landmarks into a myocardium mask
def make_heart_mask(points, shape=(256, 256)):
    points = np.asarray(points)

    # Skip missing annotations
    if points.shape != (65, 2) or np.any(points == -99999):
        return None

    inner = polygon2mask(shape, points[:32, ::-1])
    outer = polygon2mask(shape, points[33:, ::-1])

    return outer & ~inner


slice_idx = 4
time_idx = 4

image_heart = data[:, :, slice_idx, time_idx]

mask_heart = make_heart_mask(
    seg[slice_idx, time_idx],
    image_heart.shape
)

print("Myocardium mask pixels:", mask_heart.sum())

Device: cuda
Myocardium mask pixels: 777


# Manual thresholding — Heart MRI
I started with manual thresholding to see whether I could separate the myocardium from the surrounding tissues using pixel intensity, and since other parts of the image have similar intensities, I also applied thresholding inside a smaller region around the left ventricle to see if it could improve the result.

In [32]:
# Normalize the image
heart_gray = scale01(image_heart)

# Define a circular ROI around the left ventricle
cx, cy, radius = 125, 140, 43

YY, XX = np.ogrid[:heart_gray.shape[0], :heart_gray.shape[1]]

heart_roi = (XX - cx)**2 + (YY - cy)**2 <= radius**2

# Apply manual thresholding
heart_threshold = 0.25

heart_global = heart_gray > heart_threshold
heart_roi_pred = heart_global & heart_roi

# Display results
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

images_to_show = [
    image_heart,
    mask_heart,
    heart_global,
    heart_roi_pred
]

titles = [
    "Original MRI",
    "Ground truth",
    "Global threshold",
    "Threshold + ROI"
]

for ax, image, title in zip(axes, images_to_show, titles):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

# Dice scores
print(
    "Global threshold Dice:",
    round(dice_score(heart_global, mask_heart), 4)
)

print(
    "Threshold + ROI Dice:",
    round(dice_score(heart_roi_pred, mask_heart), 4)
)

<Figure size 1600x400 with 4 Axes>

Global threshold Dice: 0.023
Threshold + ROI Dice: 0.0518


# Active contours — Heart MRI
I then applied active contours because the myocardium has two boundaries, an inner one and an outer one, so I used two initial contours instead of one to see whether the algorithm could follow both boundaries and segment the region between them.

In [33]:
# Smooth the image
smoothed_heart = gaussian(heart_gray, sigma=1.5)

# Create initial contours
t = np.linspace(0, 2 * np.pi, 160, endpoint=False)

inner_init = np.column_stack([
    140 + 15 * np.sin(t),
    125 + 15 * np.cos(t)
])

outer_init = np.column_stack([
    140 + 29 * np.sin(t),
    125 + 29 * np.cos(t)
])

# Apply active contours
inner_snake = active_contour(
    smoothed_heart,
    inner_init,
    alpha=0.02,
    beta=1,
    gamma=0.01,
    max_num_iter=700
)

outer_snake = active_contour(
    smoothed_heart,
    outer_init,
    alpha=0.02,
    beta=1,
    gamma=0.01,
    max_num_iter=700
)

# Convert contours to binary masks
inner_snake_mask = polygon2mask(
    image_heart.shape, inner_snake
)

outer_snake_mask = polygon2mask(
    image_heart.shape, outer_snake
)

heart_snake_pred = outer_snake_mask & ~inner_snake_mask

# Display results
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(image_heart, cmap="gray")
axes[0].plot(
    inner_snake[:, 1],
    inner_snake[:, 0],
    "r",
    linewidth=1.5
)
axes[0].plot(
    outer_snake[:, 1],
    outer_snake[:, 0],
    "b",
    linewidth=1.5
)
axes[0].set_title("Active contour boundaries")
axes[0].axis("off")

axes[1].imshow(heart_snake_pred, cmap="gray")
axes[1].set_title("Predicted myocardium mask")
axes[1].axis("off")

plt.tight_layout()
plt.show()

# Evaluation
dice_heart_snake = dice_score(
    heart_snake_pred, mask_heart
)

print("Active contour Dice:", round(dice_heart_snake, 4))

<Figure size 1000x400 with 2 Axes>

Active contour Dice: 0.6961


# U-Net — Heart MRI
After testing thresholding and active contours, I applied U-Net as requested in the TP objectives, using the available MRI slices and temporal frames with valid annotations to create a larger training dataset.

I kept the later temporal frames for testing so I could evaluate the model on images that were not used during training, although they still come from the same MRI volume.

In [34]:
heart_items = []

# Load all valid annotated frames
for z in range(data.shape[2]):
    for t_idx in range(data.shape[3]):

        gt = make_heart_mask(
            seg[z, t_idx],
            data.shape[:2]
        )

        if gt is None:
            continue

        img = scale01(data[:, :, z, t_idx])

        # Resize image
        x = resize(
            img,
            (128, 128),
            anti_aliasing=True
        ).astype(np.float32)

        # Resize segmentation
        y = resize(
            gt.astype(np.uint8),
            (128, 128),
            order=0,
            preserve_range=True,
            anti_aliasing=False
        ).astype(np.float32)

        heart_items.append((z, t_idx, x, y))


# Training: time frames 0 to 15
heart_train = [
    item for item in heart_items
    if item[1] < 16
]

# Testing: time frames 16 to 19
heart_test = [
    item for item in heart_items
    if item[1] >= 16
]

print("Valid annotated frames:", len(heart_items))
print("Training examples:", len(heart_train))
print("Testing examples:", len(heart_test))

assert heart_train and heart_test


# Convert to tensors
def to_heart_tensors(items):
    xx = np.stack([item[2] for item in items])[:, None]
    yy = np.stack([item[3] for item in items])[:, None]

    return torch.from_numpy(xx), torch.from_numpy(yy)


hx_train, hy_train = to_heart_tensors(heart_train)
hx_test, hy_test = to_heart_tensors(heart_test)

heart_loader = DataLoader(
    TensorDataset(hx_train, hy_train),
    batch_size=8,
    shuffle=True
)

print("Training shape:", hx_train.shape)
print("Testing shape:", hx_test.shape)

Valid annotated frames: 136
Training examples: 108
Testing examples: 28
Training shape: torch.Size([108, 1, 128, 128])
Testing shape: torch.Size([28, 1, 128, 128])


In [35]:
heart_model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=1
).to(device)

heart_loss_fn = DiceLoss(sigmoid=True)

heart_optimizer = torch.optim.Adam(
    heart_model.parameters(),
    lr=0.001
)

heart_epochs = 12
heart_losses = []

for epoch in range(heart_epochs):

    heart_model.train()
    running_loss = 0

    for x_batch, y_batch in heart_loader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        heart_optimizer.zero_grad()

        outputs = heart_model(x_batch)

        loss = heart_loss_fn(outputs, y_batch)

        loss.backward()
        heart_optimizer.step()

        running_loss += loss.item() * len(x_batch)

    avg_loss = running_loss / len(heart_train)
    heart_losses.append(avg_loss)

    print(
        f"Epoch {epoch+1}/{heart_epochs}, "
        f"Loss: {avg_loss:.4f}"
    )


# Plot training loss
plt.figure(figsize=(8, 4))

plt.plot(
    range(1, heart_epochs + 1),
    heart_losses,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Dice Loss")
plt.title("Heart MRI - U-Net Training Loss")
plt.grid()
plt.show()

Epoch 1/12, Loss: 0.9697
Epoch 2/12, Loss: 0.9602
Epoch 3/12, Loss: 0.9563
Epoch 4/12, Loss: 0.9551
Epoch 5/12, Loss: 0.9543
Epoch 6/12, Loss: 0.9537
Epoch 7/12, Loss: 0.9531
Epoch 8/12, Loss: 0.9526
Epoch 9/12, Loss: 0.9520
Epoch 10/12, Loss: 0.9515
Epoch 11/12, Loss: 0.9509
Epoch 12/12, Loss: 0.9502


<Figure size 800x400 with 1 Axes>

In [36]:
heart_model.eval()

with torch.no_grad():
    predictions = []

    for batch in DataLoader(hx_test, batch_size=8):
        output = heart_model(batch.to(device))
        probability = torch.sigmoid(output)

        predictions.append(probability.cpu())

heart_prob = torch.cat(predictions).numpy()[:, 0]

heart_pred = heart_prob >= 0.5
heart_true = hy_test.numpy()[:, 0].astype(bool)

# Calculate Dice for each test image
heart_unet_dices = [
    dice_score(pred, gt)
    for pred, gt in zip(heart_pred, heart_true)
]

dice_heart_unet = float(np.mean(heart_unet_dices))

print(
    "Heart U-Net Average Dice:",
    round(dice_heart_unet, 4)
)

# Display one test example
sample = 0

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(hx_test[sample, 0], cmap="gray")
axes[0].set_title("Test MRI")

axes[1].imshow(heart_true[sample], cmap="gray")
axes[1].set_title("Ground truth")

axes[2].imshow(heart_pred[sample], cmap="gray")
axes[2].set_title("U-Net prediction")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

Heart U-Net Average Dice: 0.3577


<Figure size 1200x400 with 3 Axes>

In [37]:
methods_heart = [
    "Global threshold",
    "Threshold + ROI",
    "Active contours",
    "U-Net"
]

scores_heart = [
    dice_score(heart_global, mask_heart),
    dice_score(heart_roi_pred, mask_heart),
    dice_heart_snake,
    dice_heart_unet
]

plt.figure(figsize=(9, 5))

plt.bar(methods_heart, scores_heart)

plt.ylabel("Dice Score")
plt.title("Heart MRI - Segmentation Methods Comparison")
plt.ylim(0, 1)

plt.xticks(rotation=15)
plt.show()

for method, score in zip(methods_heart, scores_heart):
    print(f"{method}: {score:.4f}")

<Figure size 900x500 with 1 Axes>

Global threshold: 0.0230
Threshold + ROI: 0.0518
Active contours: 0.6961
U-Net: 0.3577


### Heart MRI — Observations

For the Heart MRI, I obtained very low Dice scores with global thresholding (0.0230) and thresholding with ROI (0.0518), because both methods detect other regions with similar intensities and cannot easily isolate the ring-shaped myocardium.

Active contours gave a much better result with a Dice score of 0.6961, and when I look at the predicted mask, I can see that it follows the myocardium more closely, although the boundaries are not perfect.

For U-Net, the training loss decreased from 0.9697 to 0.9502 over 12 epochs, but the average Dice score was only 0.3577, and the predicted mask still contains unwanted regions. I think the model would need more training data and further tuning to improve its segmentation.

# DRIVE

** TO DO **

**Limit of the Heart MRI comparison**

I used one selected slice and temporal frame for thresholding and active contours, while the U-Net result is the average Dice score across 28 test frames. I also separated the U-Net data by time frame, but all the images come from the same MRI volume, so this experiment does not show how the model would perform on new patients. I will therefore treat this comparison as an initial experiment rather than a definitive ranking of the methods.

In [38]:
import zipfile
from skimage import exposure

# Extract DRIVE dataset
DRIVE_path = os.path.join(Working_directory, "DRIVE")

if not os.path.isdir(
    os.path.join(DRIVE_path, "training", "images")
):
    with zipfile.ZipFile(
        os.path.join(Working_directory, "DRIVE.zip")
    ) as zip_ref:
        zip_ref.extractall(Working_directory)


# Find training images
train_im_paths = sorted(
    glob.glob(
        os.path.join(
            DRIVE_path,
            "training",
            "images",
            "*_training.tif"
        )
    )
)

print("Number of training images:", len(train_im_paths))


# Load image, vessel segmentation and retinal mask
def read_drive_case(im_path):

    name = os.path.basename(im_path).split("_")[0]

    rgb = np.array(
        Image.open(im_path).convert("RGB")
    )

    gt = np.array(
        Image.open(
            os.path.join(
                DRIVE_path,
                "training",
                "1st_manual",
                name + "_manual1.gif"
            )
        ).convert("L")
    ) > 0

    fov = np.array(
        Image.open(
            os.path.join(
                DRIVE_path,
                "training",
                "mask",
                name + "_training_mask.gif"
            )
        ).convert("L")
    ) > 0

    return rgb, gt, fov


# Load one example
rgb, vessel_gt, retinal_fov = read_drive_case(
    train_im_paths[0]
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(rgb)
axes[0].set_title("Original retinal image")

axes[1].imshow(vessel_gt, cmap="gray")
axes[1].set_title("Ground truth vessels")

axes[2].imshow(retinal_fov, cmap="gray")
axes[2].set_title("Field-of-view mask")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

Number of training images: 20


<Figure size 1300x400 with 3 Axes>

# Manual thresholding — DRIVE
I started with thresholding using the green channel of the retinal image because the blood vessels have better contrast in this channel, and I inverted the intensities so that the darker vessels would become brighter and easier to detect using a threshold.

In [39]:
# Extract green channel
retinal_green = rgb[:, :, 1].astype(np.float32) / 255.0

# Improve contrast and invert the image
enhanced = 1 - exposure.equalize_adapthist(
    retinal_green,
    clip_limit=0.02
)

# Apply manual thresholding
threshold_drive = 0.55

pred_drive_threshold = (
    enhanced > threshold_drive
) & retinal_fov

# Display results
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(enhanced, cmap="gray")
axes[0].set_title("Enhanced green channel")

axes[1].imshow(pred_drive_threshold, cmap="gray")
axes[1].set_title("Manual thresholding")

axes[2].imshow(vessel_gt, cmap="gray")
axes[2].set_title("Ground truth")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

dice_drive_threshold = dice_score(
    pred_drive_threshold,
    vessel_gt,
    retinal_fov
)

print(
    "DRIVE Threshold Dice:",
    round(dice_drive_threshold, 4)
)

<Figure size 1200x400 with 3 Axes>

DRIVE Threshold Dice: 0.4527


#Frangi vesselness filter — DRIVE
After thresholding, I applied the Frangi vesselness filter because it is designed to detect thin and elongated structures like blood vessels, so I wanted to see whether using information about their shape could give a better result than relying only on pixel intensity.

In [40]:
# Apply Frangi vesselness filter
vesselness = frangi(
    enhanced,
    sigmas=range(1, 5),
    black_ridges=False
)

# Normalize vesselness response
vesselness = scale01(vesselness)

# Automatic threshold inside the retinal area
cutoff = threshold_otsu(
    vesselness[retinal_fov]
)

pred_drive_frangi = (
    vesselness > cutoff
) & retinal_fov

# Remove small isolated regions
pred_drive_frangi = remove_small_objects(
    pred_drive_frangi,
    min_size=8
)

# Display results
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(vesselness, cmap="gray")
axes[0].set_title("Frangi vesselness")

axes[1].imshow(pred_drive_frangi, cmap="gray")
axes[1].set_title("Vessel segmentation")

axes[2].imshow(vessel_gt, cmap="gray")
axes[2].set_title("Ground truth")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

dice_drive_frangi = dice_score(
    pred_drive_frangi,
    vessel_gt,
    retinal_fov
)

print(
    "DRIVE Frangi Dice:",
    round(dice_drive_frangi, 4)
)

<Figure size 1200x400 with 3 Axes>

DRIVE Frangi Dice: 0.2904


# U-Net — DRIVE
I then applied U-Net to the DRIVE dataset using the available images and their manual vessel segmentations, and since this dataset contains more images than the Brain MRI dataset, I wanted to see whether the model could learn the vessel structures more effectively.

I used 16 images for training and kept 4 for evaluation, then resized them to 256 × 256 pixels to reduce computation, although this could make some of the thinner vessels harder to detect.

In [41]:
drive_size = (256, 256)

def preprocess_drive(im_path):

    rgb, gt, fov = read_drive_case(im_path)

    # Extract and enhance green channel
    green = rgb[:, :, 1].astype(np.float32) / 255.0

    green = exposure.equalize_adapthist(
        green,
        clip_limit=0.02
    )

    # Resize image
    x = resize(
        green,
        drive_size,
        anti_aliasing=True
    ).astype(np.float32)

    # Resize ground truth
    y = resize(
        gt.astype(np.float32),
        drive_size,
        order=0,
        anti_aliasing=False
    ).astype(np.float32)

    # Resize field-of-view mask
    m = resize(
        fov.astype(np.float32),
        drive_size,
        order=0,
        anti_aliasing=False
    ).astype(bool)

    return x, y, m


drive_preprocessed = [
    preprocess_drive(path)
    for path in train_im_paths
]

# Convert to tensors
dx = torch.from_numpy(
    np.stack([item[0] for item in drive_preprocessed])[:, None]
)

dy = torch.from_numpy(
    np.stack([item[1] for item in drive_preprocessed])[:, None]
)

dfov = np.stack([
    item[2] for item in drive_preprocessed
])

# Training: first 16 images
# Testing: last 4 images
drive_loader = DataLoader(
    TensorDataset(dx[:16], dy[:16]),
    batch_size=4,
    shuffle=True
)

print("Training:", dx[:16].shape)
print("Testing:", dx[16:].shape)

Training: torch.Size([16, 1, 256, 256])
Testing: torch.Size([4, 1, 256, 256])


In [42]:
drive_model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=1
).to(device)

drive_loss_fn = DiceLoss(sigmoid=True)

drive_optimizer = torch.optim.Adam(
    drive_model.parameters(),
    lr=0.001
)

drive_epochs = 10
drive_losses = []

for epoch in range(drive_epochs):

    drive_model.train()
    running_loss = 0

    for x_batch, y_batch in drive_loader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        drive_optimizer.zero_grad()

        outputs = drive_model(x_batch)

        loss = drive_loss_fn(outputs, y_batch)

        loss.backward()
        drive_optimizer.step()

        running_loss += loss.item() * len(x_batch)

    avg_loss = running_loss / 16
    drive_losses.append(avg_loss)

    print(
        f"Epoch {epoch+1}/{drive_epochs}, "
        f"Loss: {avg_loss:.4f}"
    )


# Display training loss
plt.figure(figsize=(8, 4))

plt.plot(
    range(1, drive_epochs + 1),
    drive_losses,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Dice Loss")
plt.title("DRIVE - U-Net Training Loss")
plt.grid()
plt.show()

Epoch 1/10, Loss: 0.8349
Epoch 2/10, Loss: 0.8183
Epoch 3/10, Loss: 0.8050
Epoch 4/10, Loss: 0.7961
Epoch 5/10, Loss: 0.7902
Epoch 6/10, Loss: 0.7858
Epoch 7/10, Loss: 0.7826
Epoch 8/10, Loss: 0.7796
Epoch 9/10, Loss: 0.7771
Epoch 10/10, Loss: 0.7749


<Figure size 800x400 with 1 Axes>

In [43]:
drive_model.eval()

with torch.no_grad():

    outputs = drive_model(dx[16:].to(device))

    drive_prob = torch.sigmoid(outputs).cpu().numpy()[:, 0]

# Binary predictions
drive_pred = drive_prob >= 0.5

drive_truth = dy[16:, 0].numpy().astype(bool)

# Dice scores for test images
drive_scores = [
    dice_score(pred, gt, fov)
    for pred, gt, fov in zip(
        drive_pred,
        drive_truth,
        dfov[16:]
    )
]

print("DRIVE U-Net Dice scores:")

for path, score in zip(train_im_paths[16:], drive_scores):
    print(os.path.basename(path), round(score, 4))

dice_drive_unet = float(np.mean(drive_scores))

print(
    "Average Dice:",
    round(dice_drive_unet, 4)
)

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(dx[16, 0], cmap="gray")
axes[0].set_title("Test retinal image")

axes[1].imshow(drive_truth[0], cmap="gray")
axes[1].set_title("Ground truth")

axes[2].imshow(drive_pred[0], cmap="gray")
axes[2].set_title("U-Net prediction")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

DRIVE U-Net Dice scores:
37_training.tif 0.2772
38_training.tif 0.2739
39_training.tif 0.2725
40_training.tif 0.2454
Average Dice: 0.2672


<Figure size 1200x400 with 3 Axes>

# Apply U-Net to an unlabeled DRIVE test image

In [44]:
test_im_paths = sorted(
    glob.glob(
        os.path.join(
            DRIVE_path,
            "test",
            "images",
            "*_test.tif"
        )
    )
)

if test_im_paths:

    test_rgb = np.array(
        Image.open(test_im_paths[0]).convert("RGB")
    )

    test_green = exposure.equalize_adapthist(
        test_rgb[:, :, 1].astype(np.float32) / 255.0,
        clip_limit=0.02
    )

    test_input = resize(
        test_green,
        drive_size,
        anti_aliasing=True
    ).astype(np.float32)

    drive_model.eval()

    with torch.no_grad():

        test_tensor = torch.from_numpy(
            test_input
        )[None, None].to(device)

        test_prob = torch.sigmoid(
            drive_model(test_tensor)
        )

    test_pred = (
        test_prob.cpu().numpy()[0, 0] >= 0.5
    )

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(test_rgb)
    axes[0].set_title("DRIVE test image")

    axes[1].imshow(test_pred, cmap="gray")
    axes[1].set_title("U-Net prediction")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

<Figure size 1000x400 with 2 Axes>

In [45]:
methods_drive = [
    "Thresholding",
    "Frangi",
    "U-Net"
]

scores_drive = [
    dice_drive_threshold,
    dice_drive_frangi,
    dice_drive_unet
]

plt.figure(figsize=(8, 5))

plt.bar(methods_drive, scores_drive)

plt.ylabel("Dice Score")
plt.title("DRIVE - Segmentation Methods Comparison")
plt.ylim(0, 1)

plt.show()

for method, score in zip(methods_drive, scores_drive):
    print(f"{method}: {score:.4f}")

<Figure size 800x500 with 1 Axes>

Thresholding: 0.4527
Frangi: 0.2904
U-Net: 0.2672


### DRIVE — Observations

I obtained a Dice score of 0.4527 with manual thresholding, and I can see that it detects many blood vessels, although it also includes regions that do not belong to the ground truth.

I expected the Frangi filter to work well because it is designed to detect thin, vessel-like structures, but I obtained a lower score of 0.2904, and I noticed that some blood vessels were missing from the result.

For U-Net, the training loss decreased from 0.8349 to 0.7749 over 10 epochs, but its average Dice score was only 0.2672, and the predicted mask is still noisy, so the model has not learned to separate the blood vessels from the background accurately.

**Limit of this comparison**

I evaluated thresholding and Frangi on one labeled training image, while the U-Net score is averaged over four held-out images, so these values are useful for discussing my experiments but they are not a fully fair comparison. I would need to evaluate all three methods on the same held-out images to compare them more reliably.

## Final conclusion

In this TP, I tested several segmentation methods on three types of medical images and compared their results using visual observations and Dice scores. I noticed that the methods do not behave in the same way depending on the structure I am trying to segment.

For the Brain MRI, thresholding with a manually selected ROI gave the best result on the image I tested, with a Dice score of 0.8463, while active contours and U-Net struggled to isolate the corpus callosum. For the Heart MRI, active contours performed best on the selected image with a Dice score of 0.6961, which makes sense because the myocardium has two boundaries that can be represented by contours. For DRIVE, manual thresholding gave the highest score among the classical methods tested on the selected image, but it still detected regions that were not blood vessels.

I also noticed that U-Net did not perform very well in these short experiments, even though the training losses decreased. I think the limited training data, number of epochs and chosen parameters contributed to these results, especially for the Brain MRI where I only had three training images.

Overall, this TP showed me that the choice of a segmentation method depends on the image and the structure being studied, and that a high score on one image does not necessarily mean the same method will work equally well on other images. To make the comparison more reliable, I would evaluate the methods on the same held-out images and test them on more data.